# JobShield – 01 Exploratory Data Analysis (Week 1)

**Group 5 · AML-3403 AI and ML Capstone**

Goal: understand the EMSCAD fake job postings dataset before building any model.

Sections: (1) load & validate · (2) class balance · (3) missing values · (4) binary flags vs fraud ·
(5) categorical fields · (6) text length · (7) top words · (8) location · (9) summary for the weekly report.

**Before running:** download `fake_job_postings.csv` from
[Kaggle](https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction) and put it in `data/raw/`.
In Google Colab, upload the file and set `DATA_PATH` below.

In [ ]:
import sys, re
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Works from the notebooks/ folder or the project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data_loader import load_emscad, combined_text, add_basic_flags, TEXT_COLS, BINARY_COLS, TARGET

DATA_PATH = ROOT / "data" / "raw" / "fake_job_postings.csv"   # change this in Colab if needed
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
REAL, FAKE = "#2E74B5", "#C55A11"
summary = {}

## 1. Load and validate

In [ ]:
df = load_emscad(DATA_PATH)
df = add_basic_flags(df)
print(f"Rows: {len(df):,}   Columns (original): {len(df.columns) - 3}")
summary["rows"] = len(df)
summary["duplicate_job_ids"] = int(df["job_id"].duplicated().sum())
print("Duplicate job_ids:", summary["duplicate_job_ids"])
df.head(3)

## 2. Class balance

In [ ]:
counts = df[TARGET].value_counts().sort_index()
pct = counts / counts.sum() * 100
summary["real"], summary["fake"] = int(counts.get(0, 0)), int(counts.get(1, 0))
summary["fake_pct"] = round(float(pct.get(1, 0)), 2)
print(pd.DataFrame({"count": counts, "percent": pct.round(2)}).rename(index={0: "Real", 1: "Fake"}))

fig, ax = plt.subplots(figsize=(5, 3.2))
bars = ax.bar(["Real", "Fake"], counts.values, color=[REAL, FAKE])
for b, p in zip(bars, pct.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height()):,}\n({p:.1f}%)",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Class balance: real vs fake postings")
ax.set_ylabel("Number of postings")
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.savefig(FIG_DIR / "01_class_balance.png", dpi=200); plt.show()

## 3. Missing values
Which fields are often empty – and are they empty more often in fake postings?

In [ ]:
cols = [c for c in df.columns if c not in ("job_id", TARGET, "no_salary", "no_company_profile", "country") + tuple(BINARY_COLS)]
miss = pd.DataFrame({
    "missing_%_all": df[cols].isna().mean() * 100,
    "missing_%_real": df[df[TARGET] == 0][cols].isna().mean() * 100,
    "missing_%_fake": df[df[TARGET] == 1][cols].isna().mean() * 100,
}).round(1).sort_values("missing_%_all", ascending=False)
summary["top_missing"] = miss["missing_%_all"].head(3).to_dict()
print(miss.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
y = range(len(miss))
ax.barh([i + 0.2 for i in y], miss["missing_%_real"], height=0.4, color=REAL, label="Real")
ax.barh([i - 0.2 for i in y], miss["missing_%_fake"], height=0.4, color=FAKE, label="Fake")
ax.set_yticks(list(y)); ax.set_yticklabels(miss.index); ax.invert_yaxis()
ax.set_xlabel("% of postings with the field missing"); ax.set_title("Missing values by class")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "02_missing_values.png", dpi=200); plt.show()

## 4. Binary flags vs fraud rate
Fraud rate = share of postings in that group that are fake.

In [ ]:
flags = BINARY_COLS + ["no_salary", "no_company_profile"]
rows = []
for f in flags:
    for v in (0, 1):
        g = df[df[f] == v]
        rows.append({"flag": f, "value": v, "postings": len(g), "fraud_rate_%": round(g[TARGET].mean() * 100, 2) if len(g) else 0})
flag_tbl = pd.DataFrame(rows)
summary["fraud_rate_no_logo"] = float(flag_tbl.query("flag=='has_company_logo' and value==0")["fraud_rate_%"].iloc[0])
summary["fraud_rate_with_logo"] = float(flag_tbl.query("flag=='has_company_logo' and value==1")["fraud_rate_%"].iloc[0])
print(flag_tbl.to_string(index=False))

piv = flag_tbl.pivot(index="flag", columns="value", values="fraud_rate_%")
fig, ax = plt.subplots(figsize=(7, 3.6))
x = range(len(piv))
ax.bar([i - 0.2 for i in x], piv[0], width=0.4, color="#9DC3E6", label="value = 0")
ax.bar([i + 0.2 for i in x], piv[1], width=0.4, color=FAKE, label="value = 1")
ax.set_xticks(list(x)); ax.set_xticklabels(piv.index, rotation=15)
ax.axhline(df[TARGET].mean() * 100, color="grey", ls="--", lw=1, label="overall fraud rate")
ax.set_ylabel("Fraud rate (%)"); ax.set_title("Fraud rate by warning-sign flag"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "03_flags_fraud_rate.png", dpi=200); plt.show()

## 5. Categorical fields

In [ ]:
def fraud_rate_by(col, min_count=50, top=10):
    g = df.groupby(df[col].fillna("Missing"))[TARGET].agg(["count", "mean"])
    g = g[g["count"] >= min_count].sort_values("mean", ascending=False).head(top)
    g["fraud_rate_%"] = (g["mean"] * 100).round(2)
    return g.drop(columns="mean")

for col in ["employment_type", "required_experience", "required_education", "function", "industry"]:
    print(f"\n=== {col} (groups with >= 50 postings, highest fraud rate first) ===")
    print(fraud_rate_by(col).to_string())

## 6. Text length by class

In [ ]:
for c in ["company_profile", "description", "requirements", "benefits"]:
    df[f"{c}_len"] = df[c].fillna("").str.split().str.len()
df["all_text_len"] = combined_text(df).str.split().str.len()
len_tbl = df.groupby(TARGET)[[f"{c}_len" for c in ["company_profile", "description", "requirements", "benefits"]] + ["all_text_len"]].median()
len_tbl.index = ["Real", "Fake"]
summary["median_words_real"], summary["median_words_fake"] = int(len_tbl.loc["Real", "all_text_len"]), int(len_tbl.loc["Fake", "all_text_len"])
print("Median number of words:"); print(len_tbl.round(0).astype(int).to_string())

fig, ax = plt.subplots(figsize=(6, 3.4))
cap = df["all_text_len"].quantile(0.99)
ax.hist(df[df[TARGET] == 0]["all_text_len"].clip(upper=cap), bins=50, alpha=0.6, density=True, color=REAL, label="Real")
ax.hist(df[df[TARGET] == 1]["all_text_len"].clip(upper=cap), bins=50, alpha=0.6, density=True, color=FAKE, label="Fake")
ax.set_xlabel("Words in combined text"); ax.set_ylabel("Density"); ax.set_title("Posting length by class"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "04_text_length.png", dpi=200); plt.show()

## 7. Top words in fake vs real postings
Simple word counts (stop words removed) – a first look before TF-IDF.

In [ ]:
STOP = set("""a an the and or of to in for on with at by from as is are be this that we you your our will
it its not have has can all any us they their who what which more new work job team company
experience skills ability including within must also well such other years year per""".split())

def top_words(texts, n=15):
    c = Counter()
    for t in texts:
        c.update(w for w in re.findall(r"[a-z]{3,}", t.lower()) if w not in STOP)
    return c.most_common(n)

text = combined_text(df)
fake_top = top_words(text[df[TARGET] == 1])
real_top = top_words(text[df[TARGET] == 0])
print(pd.DataFrame({"fake_word": [w for w, _ in fake_top], "fake_count": [n for _, n in fake_top],
                    "real_word": [w for w, _ in real_top], "real_count": [n for _, n in real_top]}).to_string(index=False))

# Words used relatively more in fake postings (min 30 fake occurrences)
cf, cr = Counter(), Counter()
for t, y in zip(text, df[TARGET]):
    (cf if y == 1 else cr).update(w for w in re.findall(r"[a-z]{3,}", t.lower()) if w not in STOP)
nf, nr = sum(cf.values()), sum(cr.values())
ratio = {w: (cf[w] / nf) / ((cr[w] + 1) / nr) for w in cf if cf[w] >= 30}
over = sorted(ratio.items(), key=lambda kv: kv[1], reverse=True)[:15]
summary["words_over_represented_in_fake"] = [w for w, _ in over[:8]]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh([w for w, _ in over][::-1], [r for _, r in over][::-1], color=FAKE)
ax.set_xlabel("How many times more frequent in fake postings"); ax.set_title("Words over-represented in fake postings")
plt.tight_layout(); plt.savefig(FIG_DIR / "05_fake_words.png", dpi=200); plt.show()

## 8. Location (country)

In [ ]:
ctry = df.groupby("country")[TARGET].agg(["count", "mean"]).sort_values("count", ascending=False).head(10)
ctry["fraud_rate_%"] = (ctry["mean"] * 100).round(2)
summary["top_country"] = ctry.index[0]
print(ctry.drop(columns="mean").to_string())

## 9. Summary for Weekly Report 1
Copy these numbers into the report.

In [ ]:
for k, v in summary.items():
    print(f"{k:32s}: {v}")